In [ ]:
import rebound, numpy as np, time
from rebound import clibrebound

def probabilistic_sticking_resolve(sim, collision):

    sim_contents = sim.contents
    p1 = sim_contents.particles[collision.p1]
    p2 = sim_contents.particles[collision.p2]

    key = frozenset((p1.name, p2.name))
    step = sim_contents.steps_done

    prev = contacts.get(key)
    if (prev is None or prev[0] < step - 1):
        stick = rng.uniform(0,1) <= prob_stick
    else:
        stick = prev[1]
    contacts[key] = (step, stick)

    clibrebound = rebound.clibrebound
    if stick:
        del contacts[key]
        return clibrebound.reb_collision_resolve_merge(sim, collision)
    return clibrebound.reb_collision_resolve_hardsphere(sim, collision)

def reflect(sim):
    for p in sim.particles:
        for q, v in (("x", "vx"), ("y", "vy"), ("z", "vz")):    # add "z","vz" for 3D
            pos, vel = getattr(p, q), getattr(p, v)
            if pos + p.r > half and vel > 0:
                setattr(p, q, half - p.r)
                setattr(p, v, -vel)
            elif pos - p.r < -half and vel < 0:
                setattr(p, q, -half + p.r)
                setattr(p, v, -vel)

sim = rebound.Simulation()
sim.units = ('m', 's', 'g')
sim.start_server(port=8000)
sim.integrator = "leapfrog"
sim.gravity = "none"                  
sim.dt = 0.4

sim.root_size = 10.0
sim.boundary = "open" 
sim.collision = "line"         
sim.collision_resolve = probabilistic_sticking_resolve

L = 10.0
half = L / 2

prob_stick = 1
rng = np.random.default_rng(42)
contacts = {}

for i in range(40):
    sim.add(m=1.0, r=0.4,
            x=rng.uniform(-half + 0.3, half - 0.3),
            y=rng.uniform(-half + 0.3, half - 0.3), z=rng.uniform(-half + 0.3, half - 0.3),
            vx=rng.normal(0, 0.2), vy=rng.normal(0, 0.2), vz=rng.normal(0, 0.2),
            name = f"p{i}")

frames = []
save_every = 5
for k, t in enumerate(np.linspace(0, 2000, 20001)):
    sim.integrate(t)
    reflect(sim)
    if k % save_every == 0:
        N = sim.N
        xyz, r = np.empty((N, 3)), np.empty(N)
        sim.serialize_particle_data(xyz=xyz, r=r)
        frames.append((sim.t, xyz, r))

REBOUND Webserver listening on http://localhost:8000 (not secure) ...
